# Librerías

In [1]:
# Librerías
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Módulos 
from astroquery.mpc import MPC
from math import ceil

# Cobs API

In [2]:
# Verificar la conexión a internet.
def verificar_conexion():
    try:
        requests.get("http://www.google.com", timeout=5)
        print('✅ Conectado a internet.')
        return True
    
    except requests.ConnectionError:
        print('🛑 Sin conexión a internet.')
        return False

In [3]:
# Conexión con la API de COBS
try:
    content = [] 
    fecha_inicial = '2000-01-01'
    nombre_cometa = 'C/2023 A3'
    Link_cops_API_pagina_1 = f'https://cobs.si/api/obs_list.api?des={nombre_cometa}&format=json&from_date={fecha_inicial}&page=1&exclude_faint=False&exclude_not_accurate=False'

    if verificar_conexion():
        print(f'\n⌛ Conectando con la base de datos [COBS Observaciones].')
        response_pagina_1 = requests.get(Link_cops_API_pagina_1)

        if response_pagina_1.status_code == 200:
            content_pagina_1 = response_pagina_1.json()
            numero_de_paginas = int(content_pagina_1['info']['pages'])

            content.extend(content_pagina_1['objects'])

            for pagina in range(2, numero_de_paginas + 1):
                Link_cops_API_pagina = f'https://cobs.si/api/obs_list.api?des={nombre_cometa}&format=json&from_date={fecha_inicial}&page={pagina}&exclude_faint=False&exclude_not_accurate=False'
                response_pagina = requests.get(Link_cops_API_pagina)
                content_pagina = response_pagina.json()
                content.extend(content_pagina['objects'])

            print('✅ Base de datos actualizada [COBS Observaciones].')
        
except requests.ConnectionError:
    print(f'🛑 Se presentó un error al cargar la base de datos.\nError: {response_pagina.status_code}\n{response_pagina.content}')

✅ Conectado a internet.

⌛ Conectando con la base de datos [COBS Observaciones].
✅ Base de datos actualizada [COBS Observaciones].


In [4]:
# Creación del data frame Cometa
cometa_df = pd.DataFrame(content)
cometa_df.__len__()

2975

In [5]:
# Numero de registros y variables sin filtrar la información
filas,columnas = cometa_df.shape
print(f'Registros: {filas}\nVariables: {columnas}')

Registros: 2975
Variables: 47


In [6]:
# Base de datos arrojada por la API
cometa_df.sample(5)

,type,obs_date,comet,observer,location,extinction,obs_method,comet_visibility,magnitude,conditions,...,magnitude_error,comparison_star_magnitude,pixel_size_x,pixel_size_y,pixel_size_unit,obs_comment,obs_sky_quality,obs_sky_quality_method,reference_star_names,date_added
48,C,2025-08-05 21:36:00,"{'type': 'C', 'name': 'C/2023 A3', 'fullname':...","{'first_name': 'Sergey', 'last_name': 'Shurpak...",None,None,"{'key': 'C', 'name': 'Unfiltered total CCD/CMO...",None,17.3,None,...,None,17.20,1.7,1.7,s,,None,NaN,,2025-08-07 20:46:25
1712,V,2024-09-22 18:43:12,"{'type': 'C', 'name': 'C/2023 A3', 'fullname':...","{'first_name': 'Robert Houston', 'last_name': ...",None,!,"{'key': 'S', 'name': 'In-Out method', 'source'...",None,4.1,None,...,None,None,None,None,None,,None,NaN,,2025-10-30 14:14:30
976,V,2024-10-22 18:45:00,"{'type': 'C', 'name': 'C/2023 A3', 'fullname':...","{'first_name': 'Peter', 'last_name': 'De Schri...",Beert,None,"{'key': 'S', 'name': 'In-Out method', 'source'...",None,4.4,None,...,None,None,None,None,None,"Through the drifting veils of clouds, still a ...",None,NaN,,2024-10-22 19:41:32
2009,V,2024-06-05 12:28:47,"{'type': 'C', 'name': 'C/2023 A3', 'fullname':...","{'first_name': 'Osamu', 'last_name': 'Miyazaki...",None,None,"{'key': 'S', 'name': 'In-Out method', 'source'...",None,10.9,None,...,None,None,None,None,None,,None,NaN,,2024-06-06 03:46:32
2869,C,2023-06-10 21:50:24,"{'type': 'C', 'name': 'C/2023 A3', 'fullname':...","{'first_name': 'Pieter-Jan', 'last_name': 'Dek...",None,None,"{'key': 'C', 'name': 'Unfiltered total CCD/CMO...",None,16.3,None,...,0.06,None,1.2,1.2,s,,None,NaN,,2023-06-14 21:52:00


In [7]:
# Métodos de observación
cometa_df.obs_method.apply(
    lambda registro: f"{registro['key']}: {registro['name']}" if (registro is not None) and ('key' in registro) and ('name' in registro) else 'Datos faltantes'
).value_counts()

obs_method
C: Unfiltered total CCD/CMOS magnitude                                              719
S: In-Out method                                                                    581
M: Modified-Out method                                                              554
Z: CCD Visual equivalent                                                            467
B: Simple Out-Out method                                                            164
V: Johnson/Bessel/Kron/Cousins V with CCD/CMOS                                      131
I: In-focus                                                                         109
k: Kron/Cousins R with CCD/CMOS                                                      89
D: Johnson/Bessel/Kron/Cousins B with CCD/CMOS                                       27
P: photographic                                                                      24
H: Kron/Cousins I with CCD/CMOS                                                      21
-: Unknown           

In [8]:
# Tratamiento de los datos de interés
cometa_df['obs_method_key'] = cometa_df.obs_method.apply(lambda registro: registro['key'] if registro is not None and 'key' in registro else 'Dato faltante')
cometa_df['obs_date'] = pd.to_datetime(pd.to_datetime(cometa_df.obs_date).dt.date)
cometa_df['magnitude'] = pd.to_numeric(cometa_df.magnitude)

In [9]:
# Creación del data frame curva de luz cruda
curva_de_luz_cruda_df = cometa_df[['obs_method_key', 'obs_date', 'magnitude']].copy()
curva_de_luz_cruda_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2975 entries, 0 to 2974
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   obs_method_key  2975 non-null   object        
 1   obs_date        2975 non-null   datetime64[ns]
 2   magnitude       2965 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 69.9+ KB


In [10]:
# Numero de registros y variables con la información filtrada
filas,columnas = curva_de_luz_cruda_df.shape
print(f'Registros: {filas}\nVariables: {columnas}')

Registros: 2975
Variables: 3


In [11]:
# Data Frame de la curva de luz
curva_de_luz_cruda_df.sample(5)

,obs_method_key,obs_date,magnitude
1877,S,2024-06-28,10.8
971,P,2024-10-22,4.2
691,M,2024-11-02,6.6
1850,M,2024-07-02,10.0
555,S,2024-11-10,8.5


In [12]:
# Curva de luz cruda.
labels = {'obs_date':'Observation Date','magnitude':'Apparent total magnitude', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_cruda_df, x='obs_date', y='magnitude', color='obs_method_key', template= 'plotly_dark', labels= labels, title= f'Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

# Perihelio Cobs API

In [13]:
# Conexión con la API de COBS para obtener el perihelio
try: 
    Link_cops_API = f'https://cobs.si/api/comet.api?des={nombre_cometa}'

    if verificar_conexion():
        response = requests.get(Link_cops_API)

        if response.status_code == 200:
            perihelio = pd.to_datetime(response.json()['object']['perihelion_date'])
            print('✅ Perihelio del cometa obtenido.')
    
except requests.ConnectionError:
    print(f'🛑 Se presentó un error al cargar la base de datos.\nError: {response.status_code}\n{response.content}')

✅ Conectado a internet.
✅ Perihelio del cometa obtenido.


# MPC API usando astroquery.

In [14]:
# Creación de data frame Ephemeris (conexión con la API del MPC)
efemerides_total = []

fecha_inicial = curva_de_luz_cruda_df.obs_date.min()
fecha_final = curva_de_luz_cruda_df.obs_date.max()
fechas = (fecha_final - fecha_inicial).days + 1

print('⌛ Conectando con la base de datos [MPC efemerides].')
for i in range(ceil(fechas/1441)):
    efemerides = MPC.get_ephemeris(nombre_cometa, start = str(fecha_inicial), number = 1441)  # type: ignore
    efemerides_ciclo_df = efemerides.to_pandas()
    efemerides_total.append(efemerides_ciclo_df)

    fecha_inicial = efemerides_ciclo_df.Date.max()

# Creación del data frame efemerides filtrada
efemerides_df = pd.concat(efemerides_total)
efemerides_df.columns = efemerides_df.columns.str.lower().str.replace(' ', '_')

efemerides_filtrada_df = efemerides_df[['date', 'delta','r', 'phase']].copy()
efemerides_filtrada_df = efemerides_filtrada_df.rename(columns = {'date':'obs_date'})
efemerides_filtrada_df['obs_date'] = pd.to_datetime(pd.to_datetime(efemerides_filtrada_df.obs_date).dt.date)
efemerides_filtrada_df.reset_index(inplace = True)

efemerides_filtrada_df

⌛ Conectando con la base de datos [MPC efemerides].


,index,obs_date,delta,r,phase
0,0,2023-02-25,6.946,7.290,7.5
1,1,2023-02-26,6.921,7.281,7.4
2,2,2023-02-27,6.896,7.273,7.4
3,3,2023-02-28,6.872,7.264,7.4
4,4,2023-03-01,6.847,7.255,7.3
...,...,...,...,...,...
1436,1436,2027-01-31,10.168,9.541,4.4
1437,1437,2027-02-01,10.168,9.548,4.5
1438,1438,2027-02-02,10.167,9.556,4.5
1439,1439,2027-02-03,10.166,9.564,4.5


In [15]:
# Info del data frame ephemeris
efemerides_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1441 entries, 0 to 1440
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           1441 non-null   datetime64[ns]
 1   ra             1441 non-null   float64       
 2   dec            1441 non-null   float64       
 3   delta          1441 non-null   float64       
 4   r              1441 non-null   float64       
 5   elongation     1441 non-null   float64       
 6   phase          1441 non-null   float64       
 7   v              1441 non-null   float64       
 8   proper_motion  1441 non-null   float64       
 9   direction      1441 non-null   float64       
dtypes: datetime64[ns](1), float64(9)
memory usage: 112.7 KB


In [16]:
# Dar a los datos el formato deseado
efemerides_df.date = pd.to_datetime(efemerides_df.date)
efemerides_df.date = pd.to_datetime(efemerides_df.date.dt.date)
efemerides_df.dtypes

date             datetime64[ns]
ra                      float64
dec                     float64
delta                   float64
r                       float64
elongation              float64
phase                   float64
v                       float64
proper_motion           float64
direction               float64
dtype: object

In [17]:
# Creación del data frame ephemeris filtrada
efemerides_filtrada_df = efemerides_df[['date', 'delta','r', 'phase']].copy()
efemerides_filtrada_df = efemerides_filtrada_df.rename(columns = {'date':'obs_date'})
efemerides_filtrada_df

,obs_date,delta,r,phase
0,2023-02-25,6.946,7.290,7.5
1,2023-02-26,6.921,7.281,7.4
2,2023-02-27,6.896,7.273,7.4
3,2023-02-28,6.872,7.264,7.4
4,2023-03-01,6.847,7.255,7.3
...,...,...,...,...
1436,2027-01-31,10.168,9.541,4.4
1437,2027-02-01,10.168,9.548,4.5
1438,2027-02-02,10.167,9.556,4.5
1439,2027-02-03,10.166,9.564,4.5


# Unión de las bases de datos.

In [18]:
# Unión de las bases de datos COBS y MPC
curva_de_luz_procesada_df = curva_de_luz_cruda_df.merge(efemerides_filtrada_df, on='obs_date')
curva_de_luz_procesada_df

,obs_method_key,obs_date,magnitude,delta,r,phase
0,V,2025-11-18,18.0,6.355,5.769,7.5
1,C,2025-10-24,17.9,5.847,5.522,9.5
2,C,2025-10-20,17.4,5.759,5.482,9.8
3,C,2025-10-19,17.8,5.736,5.472,9.8
4,C,2025-10-18,17.9,5.714,5.462,9.9
...,...,...,...,...,...,...
2970,C,2023-02-27,18.0,6.896,7.273,7.4
2971,C,2023-02-26,18.1,6.921,7.281,7.4
2972,C,2023-02-25,18.2,6.946,7.290,7.5
2973,C,2023-02-25,17.9,6.946,7.290,7.5


In [19]:
# Información del data frame curva de lus procesada
curva_de_luz_procesada_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2975 entries, 0 to 2974
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   obs_method_key  2975 non-null   object        
 1   obs_date        2975 non-null   datetime64[ns]
 2   magnitude       2965 non-null   float64       
 3   delta           2975 non-null   float64       
 4   r               2975 non-null   float64       
 5   phase           2975 non-null   float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 139.6+ KB


In [20]:
# Reducción de la magnitud aparente y calculo del Delta t
beta = 0

curva_de_luz_procesada_df['magnitud_reducida'] = (
    curva_de_luz_cruda_df['magnitude'] 
    - 5 * np.log10(curva_de_luz_procesada_df['delta'] * curva_de_luz_procesada_df['r'])
    - (beta * curva_de_luz_procesada_df['phase'])
    )

curva_de_luz_procesada_df['delta_t'] = (curva_de_luz_procesada_df.obs_date - perihelio) # type: ignore
curva_de_luz_procesada_df['delta_t'] = curva_de_luz_procesada_df.delta_t.apply(lambda delta_t: delta_t.days)

curva_de_luz_procesada_df

,obs_method_key,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,V,2025-11-18,18.0,6.355,5.769,7.5,10.178920,416
1,C,2025-10-24,17.9,5.847,5.522,9.5,10.354853,391
2,C,2025-10-20,17.4,5.759,5.482,9.8,9.903569,387
3,C,2025-10-19,17.8,5.736,5.472,9.8,10.316224,386
4,C,2025-10-18,17.9,5.714,5.462,9.9,10.428540,385
...,...,...,...,...,...,...,...,...
2970,C,2023-02-27,18.0,6.896,7.273,7.4,9.498446,-579
2971,C,2023-02-26,18.1,6.921,7.281,7.4,9.588201,-580
2972,C,2023-02-25,18.2,6.946,7.290,7.5,9.677688,-581
2973,C,2023-02-25,17.9,6.946,7.290,7.5,9.377688,-581


In [21]:
# Curva de luz reducida
labels = {'obs_date':'Observation Date','magnitud_reducida':'Apparent total magnitude processed', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_procesada_df, x='obs_date', y='magnitud_reducida', color='obs_method_key', template= 'plotly_dark', labels= labels, title=f'Reduced Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [22]:
# Curva de luz reducida
labels = {'delta_t':'t-Δt','magnitud_reducida':'Apparent total magnitude processed', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_procesada_df, x='delta_t', y='magnitud_reducida', color='obs_method_key', template= 'plotly_dark', labels= labels, title=f'Reduced Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [23]:
# Creación del data frame curva de luz promediada
numero_elementos_grupo = 9

curva_de_luz_promediada_df = curva_de_luz_procesada_df.copy()
curva_de_luz_promediada_df['promedio_movil'] = curva_de_luz_promediada_df.magnitud_reducida.rolling(window = numero_elementos_grupo).mean()
curva_de_luz_promediada_df

,obs_method_key,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,V,2025-11-18,18.0,6.355,5.769,7.5,10.178920,416,NaN
1,C,2025-10-24,17.9,5.847,5.522,9.5,10.354853,391,NaN
2,C,2025-10-20,17.4,5.759,5.482,9.8,9.903569,387,NaN
3,C,2025-10-19,17.8,5.736,5.472,9.8,10.316224,386,NaN
4,C,2025-10-18,17.9,5.714,5.462,9.9,10.428540,385,NaN
...,...,...,...,...,...,...,...,...,...
2970,C,2023-02-27,18.0,6.896,7.273,7.4,9.498446,-579,9.492396
2971,C,2023-02-26,18.1,6.921,7.281,7.4,9.588201,-580,9.506195
2972,C,2023-02-25,18.2,6.946,7.290,7.5,9.677688,-581,9.529937
2973,C,2023-02-25,17.9,6.946,7.290,7.5,9.377688,-581,9.511869


In [24]:
# Curva de luz Promediada
labels = {'obs_date':'Observation Date','magnitud_reducida':'Max apparent total magnitude reduced', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_promediada_df, x='obs_date', y='promedio_movil', color='obs_method_key', template= 'plotly_dark', labels= labels, title= f'Average Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [25]:
# Curva de luz Promediada
labels = {'delta_t':'t - Δt','magnitud_reducida':'Max apparent total magnitude reduced', 'obs_method_key' : 'Observation Method'}
fig = px.scatter(curva_de_luz_promediada_df, x='delta_t', y='promedio_movil', color='obs_method_key', template= 'plotly_dark', labels= labels, title= f'Average Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz interna (Envolvente inferior) v1 (Promedio corrido -> agrupación)

In [26]:
# Creación del data frame curva de luz agrupada
curva_de_luz_interna_v1_df = curva_de_luz_promediada_df.groupby(by = 'obs_date').max()
curva_de_luz_interna_v1_df = curva_de_luz_interna_v1_df.reset_index()

curva_de_luz_interna_v1_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2023-02-25,C,18.3,6.946,7.290,7.5,9.777688,-581,9.529937
1,2023-02-26,C,18.1,6.921,7.281,7.4,9.588201,-580,9.506195
2,2023-02-27,C,18.0,6.896,7.273,7.4,9.498446,-579,9.523024
3,2023-02-28,C,18.2,6.872,7.264,7.4,9.708705,-578,9.531429
4,2023-03-01,Z,17.9,6.847,7.255,7.3,9.419311,-577,9.528171
...,...,...,...,...,...,...,...,...,...
558,2025-10-18,C,17.9,5.714,5.462,9.9,10.428540,385,NaN
559,2025-10-19,C,17.8,5.736,5.472,9.8,10.316224,386,NaN
560,2025-10-20,C,17.4,5.759,5.482,9.8,9.903569,387,NaN
561,2025-10-24,C,17.9,5.847,5.522,9.5,10.354853,391,NaN


In [27]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v1_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [28]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v1_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz Externa (Envolvente superior) v1 (Promedio corrido -> agrupación)

In [29]:
# Creación del data frame curva de luz agrupada
curva_de_luz_externa_v1_df = curva_de_luz_promediada_df.groupby(by = 'obs_date').min()
curva_de_luz_externa_v1_df = curva_de_luz_externa_v1_df.reset_index()
curva_de_luz_externa_v1_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2023-02-25,C,17.9,6.946,7.290,7.5,9.377688,-581,9.511869
1,2023-02-26,C,18.1,6.921,7.281,7.4,9.588201,-580,9.506195
2,2023-02-27,C,17.9,6.896,7.273,7.4,9.398446,-579,9.492396
3,2023-02-28,C,17.7,6.872,7.264,7.4,9.208705,-578,9.518689
4,2023-03-01,Z,17.9,6.847,7.255,7.3,9.419311,-577,9.528171
...,...,...,...,...,...,...,...,...,...
558,2025-10-18,C,17.9,5.714,5.462,9.9,10.428540,385,NaN
559,2025-10-19,C,17.8,5.736,5.472,9.8,10.316224,386,NaN
560,2025-10-20,C,17.4,5.759,5.482,9.8,9.903569,387,NaN
561,2025-10-24,C,17.9,5.847,5.522,9.5,10.354853,391,NaN


In [30]:
# Gráfica de lus promediada
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v1_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=6, line= dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [31]:
# Gráfica de lus promediada
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v1_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=6, line= dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz interna (Envolvente inferior) v2 (Agrupación  -> promedio corrido)

In [32]:
curva_de_luz_procesada_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2975 entries, 0 to 2974
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   obs_method_key     2975 non-null   object        
 1   obs_date           2975 non-null   datetime64[ns]
 2   magnitude          2965 non-null   float64       
 3   delta              2975 non-null   float64       
 4   r                  2975 non-null   float64       
 5   phase              2975 non-null   float64       
 6   magnitud_reducida  2965 non-null   float64       
 7   delta_t            2975 non-null   int64         
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 186.1+ KB


In [32]:
# Creación del data frame curva de luz agrupada
curva_de_luz_agrupada_max_v2_df = curva_de_luz_procesada_df.groupby(by = 'obs_date').max()
curva_de_luz_agrupada_max_v2_df = curva_de_luz_agrupada_max_v2_df.reset_index()
curva_de_luz_agrupada_max_v2_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,2023-02-25,C,18.3,6.946,7.290,7.5,9.777688,-581
1,2023-02-26,C,18.1,6.921,7.281,7.4,9.588201,-580
2,2023-02-27,C,18.0,6.896,7.273,7.4,9.498446,-579
3,2023-02-28,C,18.2,6.872,7.264,7.4,9.708705,-578
4,2023-03-01,Z,17.9,6.847,7.255,7.3,9.419311,-577
...,...,...,...,...,...,...,...,...
558,2025-10-18,C,17.9,5.714,5.462,9.9,10.428540,385
559,2025-10-19,C,17.8,5.736,5.472,9.8,10.316224,386
560,2025-10-20,C,17.4,5.759,5.482,9.8,9.903569,387
561,2025-10-24,C,17.9,5.847,5.522,9.5,10.354853,391


In [33]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_agrupada_max_v2_df, x='obs_date', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Min Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [34]:
# Creación del data frame curva de luz promediada
numero_elementos_grupo = 7

curva_de_luz_interna_v2_df = curva_de_luz_agrupada_max_v2_df.copy()
curva_de_luz_interna_v2_df['promedio_movil'] = curva_de_luz_interna_v2_df.magnitud_reducida.rolling(window = numero_elementos_grupo, center= True).mean()
curva_de_luz_interna_v2_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2023-02-25,C,18.3,6.946,7.290,7.5,9.777688,-581,NaN
1,2023-02-26,C,18.1,6.921,7.281,7.4,9.588201,-580,NaN
2,2023-02-27,C,18.0,6.896,7.273,7.4,9.498446,-579,NaN
3,2023-02-28,C,18.2,6.872,7.264,7.4,9.708705,-578,9.608898
4,2023-03-01,Z,17.9,6.847,7.255,7.3,9.419311,-577,9.564086
...,...,...,...,...,...,...,...,...,...
558,2025-10-18,C,17.9,5.714,5.462,9.9,10.428540,385,10.461229
559,2025-10-19,C,17.8,5.736,5.472,9.8,10.316224,386,10.387666
560,2025-10-20,C,17.4,5.759,5.482,9.8,9.903569,387,NaN
561,2025-10-24,C,17.9,5.847,5.522,9.5,10.354853,391,NaN


In [35]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v2_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [36]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v2_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz Externa (Envolvente superior) v2 (Agrupación  -> promedio corrido)

In [37]:
# Creación del data frame curva de luz agrupada
curva_de_luz_agrupada_min_v2_df = curva_de_luz_procesada_df.groupby(by = 'obs_date').min()
curva_de_luz_agrupada_min_v2_df = curva_de_luz_agrupada_min_v2_df.reset_index()
curva_de_luz_agrupada_min_v2_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,2023-02-25,C,17.9,6.946,7.290,7.5,9.377688,-581
1,2023-02-26,C,18.1,6.921,7.281,7.4,9.588201,-580
2,2023-02-27,C,17.9,6.896,7.273,7.4,9.398446,-579
3,2023-02-28,C,17.7,6.872,7.264,7.4,9.208705,-578
4,2023-03-01,Z,17.9,6.847,7.255,7.3,9.419311,-577
...,...,...,...,...,...,...,...,...
558,2025-10-18,C,17.9,5.714,5.462,9.9,10.428540,385
559,2025-10-19,C,17.8,5.736,5.472,9.8,10.316224,386
560,2025-10-20,C,17.4,5.759,5.482,9.8,9.903569,387
561,2025-10-24,C,17.9,5.847,5.522,9.5,10.354853,391


In [38]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_agrupada_min_v2_df, x='obs_date', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [39]:
# Creación del data frame curva de luz promediada
numero_elementos_grupo = 7

curva_de_luz_externa_v2_df = curva_de_luz_agrupada_min_v2_df.copy()
curva_de_luz_externa_v2_df['promedio_movil'] = curva_de_luz_externa_v2_df.magnitud_reducida.rolling(window = numero_elementos_grupo, center= True).mean()
curva_de_luz_externa_v2_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2023-02-25,C,17.9,6.946,7.290,7.5,9.377688,-581,NaN
1,2023-02-26,C,18.1,6.921,7.281,7.4,9.588201,-580,NaN
2,2023-02-27,C,17.9,6.896,7.273,7.4,9.398446,-579,NaN
3,2023-02-28,C,17.7,6.872,7.264,7.4,9.208705,-578,9.466041
4,2023-03-01,Z,17.9,6.847,7.255,7.3,9.419311,-577,9.478372
...,...,...,...,...,...,...,...,...,...
558,2025-10-18,C,17.9,5.714,5.462,9.9,10.428540,385,10.232657
559,2025-10-19,C,17.8,5.736,5.472,9.8,10.316224,386,10.159094
560,2025-10-20,C,17.4,5.759,5.482,9.8,9.903569,387,NaN
561,2025-10-24,C,17.9,5.847,5.522,9.5,10.354853,391,NaN


In [40]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v2_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [41]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v2_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz usando la mediada v1 (Mediana de cada día registrado).

In [42]:
# Creación del data frame curva de luz mediana v1
curva_de_luz_mediada_v1_df = curva_de_luz_procesada_df.groupby(by= 'obs_date').median(numeric_only= True)
curva_de_luz_mediada_v1_df = curva_de_luz_mediada_v1_df.reset_index()
curva_de_luz_mediada_v1_df

,obs_date,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,2023-02-25,18.20,6.946,7.290,7.5,9.677688,-581.0
1,2023-02-26,18.10,6.921,7.281,7.4,9.588201,-580.0
2,2023-02-27,17.95,6.896,7.273,7.4,9.448446,-579.0
3,2023-02-28,17.95,6.872,7.264,7.4,9.458705,-578.0
4,2023-03-01,17.90,6.847,7.255,7.3,9.419311,-577.0
...,...,...,...,...,...,...,...
558,2025-10-18,17.90,5.714,5.462,9.9,10.428540,385.0
559,2025-10-19,17.80,5.736,5.472,9.8,10.316224,386.0
560,2025-10-20,17.40,5.759,5.482,9.8,9.903569,387.0
561,2025-10-24,17.90,5.847,5.522,9.5,10.354853,391.0


In [43]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v1_df, x='obs_date', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [44]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v1_df, x='delta_t', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz usando la mediada v2 (Mediana de las dos curvas).

In [45]:
# Creación del data frame curva de luz mediana v2
curva_de_luz_mediada_v2_df = curva_de_luz_externa_v2_df.copy()
curva_de_luz_mediada_v2_df['mediana'] = (curva_de_luz_interna_v2_df['promedio_movil'] + curva_de_luz_externa_v2_df['promedio_movil'])/2
curva_de_luz_mediada_v2_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil,mediana
0,2023-02-25,C,17.9,6.946,7.290,7.5,9.377688,-581,NaN,NaN
1,2023-02-26,C,18.1,6.921,7.281,7.4,9.588201,-580,NaN,NaN
2,2023-02-27,C,17.9,6.896,7.273,7.4,9.398446,-579,NaN,NaN
3,2023-02-28,C,17.7,6.872,7.264,7.4,9.208705,-578,9.466041,9.537469
4,2023-03-01,Z,17.9,6.847,7.255,7.3,9.419311,-577,9.478372,9.521229
...,...,...,...,...,...,...,...,...,...,...
558,2025-10-18,C,17.9,5.714,5.462,9.9,10.428540,385,10.232657,10.346943
559,2025-10-19,C,17.8,5.736,5.472,9.8,10.316224,386,10.159094,10.273380
560,2025-10-20,C,17.4,5.759,5.482,9.8,9.903569,387,NaN,NaN
561,2025-10-24,C,17.9,5.847,5.522,9.5,10.354853,391,NaN,NaN


In [46]:
# Gráfica de luz interna
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v2_df, x='obs_date', y='mediana', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

In [47]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v2_df, x='delta_t', y='mediana', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Comparación de las curvas de luz v1 (Promedio corrido -> agrupación)

In [48]:
# Gráfica de luz promediada
labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_externa_v1_df.obs_date, y=curva_de_luz_externa_v1_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v1_df.obs_date, y=curva_de_luz_interna_v1_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_mediada_v1_df.obs_date, y=curva_de_luz_mediada_v1_df.magnitud_reducida, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_externa_v2_df.obs_date, y=curva_de_luz_externa_v2_df.promedio_movil, mode='markers', name='Envolvente_v2', marker=dict(color='green', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_interna_v1_df.obs_date, y=curva_de_luz_interna_v2_df.promedio_movil, mode='markers', name='Núcleo_v2', marker=dict(color='blue', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_mediada_v1_df.obs_date, y=curva_de_luz_mediada_v2_df.mediana, mode='markers', name='Mediana_v2', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='Observation Date', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')
fig.show()

In [49]:
# Gráfica de luz promediada
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_externa_v1_df.delta_t, y=curva_de_luz_externa_v1_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v1_df.delta_t, y=curva_de_luz_interna_v1_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_mediada_v1_df.delta_t, y=curva_de_luz_mediada_v1_df.magnitud_reducida, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='t - Δt', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')
fig.show()

# Comparación de las curvas de luz v2 (Agrupación -> promedio corrido)

In [50]:
# Gráfica de luz promediada
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_agrupada_min_v2_df.obs_date, y=curva_de_luz_agrupada_min_v2_df.magnitud_reducida, mode='markers', name='máximo diario', marker=dict(color="#fa00e9", line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_agrupada_max_v2_df.obs_date, y=curva_de_luz_agrupada_max_v2_df.magnitud_reducida, mode='markers', name='minimo diario', marker=dict(color="#02FA61", line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_externa_v2_df.obs_date, y=curva_de_luz_externa_v2_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.obs_date, y=curva_de_luz_interna_v2_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.obs_date, y=curva_de_luz_mediada_v2_df.mediana, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))

fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='Observation Date', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')

fig.show()

# Envolvente superior calculada con percentiles v3

In [51]:
# Creación del data frame curva de luz agrupada
curva_de_luz_agrupada_min_v3_df = curva_de_luz_procesada_df.groupby(by = 'obs_date').min()
curva_de_luz_agrupada_min_v3_df = curva_de_luz_agrupada_min_v3_df.reset_index()
curva_de_luz_agrupada_min_v3_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t
0,2023-02-25,C,17.9,6.946,7.290,7.5,9.377688,-581
1,2023-02-26,C,18.1,6.921,7.281,7.4,9.588201,-580
2,2023-02-27,C,17.9,6.896,7.273,7.4,9.398446,-579
3,2023-02-28,C,17.7,6.872,7.264,7.4,9.208705,-578
4,2023-03-01,Z,17.9,6.847,7.255,7.3,9.419311,-577
...,...,...,...,...,...,...,...,...
558,2025-10-18,C,17.9,5.714,5.462,9.9,10.428540,385
559,2025-10-19,C,17.8,5.736,5.472,9.8,10.316224,386
560,2025-10-20,C,17.4,5.759,5.482,9.8,9.903569,387
561,2025-10-24,C,17.9,5.847,5.522,9.5,10.354853,391


In [52]:
curva_de_luz_interna_v3_df = curva_de_luz_agrupada_min_v3_df.copy()
curva_de_luz_interna_v3_df['percentil_movil'] = curva_de_luz_agrupada_min_v3_df.magnitud_reducida.rolling(window=3, center=True).apply(
    lambda x: np.percentile(x, 0), raw=True)
curva_de_luz_externa_v2_df

,obs_date,obs_method_key,magnitude,delta,r,phase,magnitud_reducida,delta_t,promedio_movil
0,2023-02-25,C,17.9,6.946,7.290,7.5,9.377688,-581,NaN
1,2023-02-26,C,18.1,6.921,7.281,7.4,9.588201,-580,NaN
2,2023-02-27,C,17.9,6.896,7.273,7.4,9.398446,-579,NaN
3,2023-02-28,C,17.7,6.872,7.264,7.4,9.208705,-578,9.466041
4,2023-03-01,Z,17.9,6.847,7.255,7.3,9.419311,-577,9.478372
...,...,...,...,...,...,...,...,...,...
558,2025-10-18,C,17.9,5.714,5.462,9.9,10.428540,385,10.232657
559,2025-10-19,C,17.8,5.736,5.472,9.8,10.316224,386,10.159094
560,2025-10-20,C,17.4,5.759,5.482,9.8,9.903569,387,NaN
561,2025-10-24,C,17.9,5.847,5.522,9.5,10.354853,391,NaN


In [53]:
# Gráfica de luz promediada
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_procesada_df.obs_date, y=curva_de_luz_procesada_df.magnitud_reducida, mode='markers', name='magnitud reducida', marker=dict(line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_agrupada_min_v3_df.obs_date, y=curva_de_luz_agrupada_min_v3_df.magnitud_reducida, mode='markers', name='máximo diario', marker=dict(color="#fa00e9", line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v3_df.obs_date, y=curva_de_luz_interna_v3_df.percentil_movil, mode='markers', name='Envolvente Percentil', marker=dict(color="#02c7fe", line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.obs_date, y=curva_de_luz_externa_v2_df.promedio_movil, mode='markers', name='Envolvente Promedio', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))

fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='Observation Date', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')

fig.show()

# Guardar datos

In [54]:
curva_de_luz_procesada_df.to_csv(r'Bases_de_datos/curva_de_luz_procesada_COBS.txt', index=False)
curva_de_luz_interna_v2_df.to_csv(r'Bases_de_datos/curva_de_luz_interna_COBS.txt', index=False)
curva_de_luz_externa_v2_df.to_csv(r'Bases_de_datos/curva_de_luz_externa_COBS.txt', index=False)